<a href="https://colab.research.google.com/github/noname123920/diplom/blob/main/pinn_deepxde2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install deepxde

In [2]:
import os
os.environ["DDE_BACKEND"] = "pytorch"
import deepxde as dde
import numpy as np
import torch
import plotly.express as px
import itertools

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


In [3]:
dde.config.set_default_float("float64")

Set the default float type to float64


In [4]:
kappa = 1.0
geom = dde.geometry.Interval(0, np.pi)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

In [5]:
def pde(x, p):
    dp_t = dde.grad.jacobian(p, x, i=0, j=1)
    dp_xx = dde.grad.hessian(p, x, i=0, j=0)
    return dp_t - kappa * dp_xx

In [6]:
def exact(x):
    xx = x[:, 0:1]
    tt = x[:, 1:2]
    return np.exp(-tt) * np.sin(xx)

In [7]:
iterations = 10000
epsilon = 1e-4

In [8]:
def output(x, y):
    x_col = x[:, 0:1]
    t_col = x[:, 1:2]
    return torch.sin(x_col) + x_col * (np.pi - x_col) * t_col * y

In [9]:
XT_test = geomtime.uniform_points(5000)
y_true = exact(XT_test)

In [10]:
fig = px.scatter(
    x=XT_test[:, 0],
    y=XT_test[:, 1],
    labels={"x": "Координата x", "y": "Время t"}
)

fig.add_shape(
    type="rect",
    x0=XT_test[:, 0].min(), y0=XT_test[:, 1].min(),
    x1=XT_test[:, 0].max(), y1=XT_test[:, 1].max(),
    line=dict(color="blue", width=2),
    fillcolor="rgba(0, 0, 255, 0.05)",
)

fig.update_layout(template="plotly_white")
fig.update_traces(marker=dict(size=6, color="red"))
fig.show()

In [11]:
search_space = {"layers": list(range(2, 8, 2)), "neurons": list(range(20, 60, 10)), "num_domain": list(range(100, 5000, 200))}
keys = search_space.keys()
combinations = itertools.product(*search_space.values())

In [12]:
dde.optimizers.config.set_LBFGS_options(maxiter=5000)
for values in combinations:
  params = dict(zip(keys, values))

  # dde.config.set_random_seed(42)

  data = dde.data.TimePDE(geomtime, pde, [], num_domain=params["num_domain"], num_test=5000, solution=exact, train_distribution="uniform")

  net = dde.nn.FNN([2] + [params["neurons"]] * params["layers"] + [1], "tanh", "Glorot normal")
  net.apply_output_transform(output)

  model = dde.Model(data, net)
  model.compile("adam", lr=1e-3, metrics=["l2 relative error"])
  model.train(iterations=iterations)

  model.compile("L-BFGS", metrics=["l2 relative error"])
  model.train()

  y_pred = model.predict(XT_test)
  l2_error = dde.metrics.l2_relative_error(y_true, y_pred)

  print(f"{params} | L2 Error: {l2_error:.4e}")

  if l2_error < epsilon:
    break

Compiling model...
'compile' took 4.923148 s

Training model...

Step      Train loss    Test loss     Test metric   
0         [1.87e+00]    [1.47e+00]    [6.54e-01]    
1000      [5.48e-04]    [8.18e-04]    [1.14e-02]    
2000      [1.33e-04]    [2.10e-04]    [5.04e-03]    
3000      [3.53e-05]    [5.56e-05]    [1.76e-03]    
4000      [1.55e-05]    [2.45e-05]    [1.04e-03]    
5000      [9.28e-06]    [1.50e-05]    [9.46e-04]    
6000      [7.18e-06]    [1.20e-05]    [9.19e-04]    
7000      [5.90e-06]    [1.05e-05]    [9.10e-04]    
8000      [4.94e-06]    [9.41e-06]    [8.96e-04]    
9000      [4.19e-06]    [8.49e-06]    [8.81e-04]    
10000     [3.61e-06]    [7.72e-06]    [8.65e-04]    

Best model at step 10000:
  train loss: 3.61e-06
  test loss: 7.72e-06
  test metric: [8.65e-04]

'train' took 51.472598 s

Compiling model...
'compile' took 0.000534 s

Training model...

Step      Train loss    Test loss     Test metric   
10000     [3.61e-06]    [7.72e-06]    [8.65e-04]    
110